In [2]:
from __future__ import annotations

import hashlib
import os
import re
from dataclasses import dataclass
from pathlib import Path


# ---- Config ----
# If you launched the notebook from the repo root, this should already be correct.
ROOT = Path.cwd().joinpath("..")

# What files to scan
HEADER_EXTS = {".h", ".hpp", ".hh", ".hxx", ".inl"}

# Folders to skip
SKIP_DIR_NAMES = {
    ".git",
    ".venv",
    "venv",
    "build",
    "out",
    "dist",
    "third_party",
    "external",
    "packages",
    ".xmake",
}

# Prefix for include guards
GUARD_PREFIX = "VULKANENGINE_"


PRAGMA_ONCE_RE = re.compile(r"(?m)^\s*#\s*pragma\s+once\b.*$")


def looks_like_repo_root(path: Path) -> bool:
    return (path / "xmake.lua").exists() and (path / "include").exists()


if not looks_like_repo_root(ROOT):
    raise RuntimeError(
        f"""ROOT={ROOT} doesn't look like the VulkanEngine repo root. \
,
Open the notebook from the repo root or set ROOT manually."""
    )

In [3]:
@dataclass(frozen=True)
class Candidate:
    path: Path
    rel: str
    guard: str


def _sanitize_guard_token(token: str) -> str:
    # Replace separators and non-identifier chars with underscores
    token = re.sub(r"[\\/\. ]+", "_", token)
    token = re.sub(r"[^A-Za-z0-9_]", "_", token)
    token = re.sub(r"_+", "_", token)
    return token.strip("_")


def compute_guard(rel_path: str) -> str:
    base = _sanitize_guard_token(rel_path)
    return (GUARD_PREFIX + base).upper()


def short_hash(text: str, n: int = 8) -> str:
    return hashlib.sha1(text.encode("utf-8")).hexdigest()[:n].upper()


def detect_newline_style(text: str) -> str:
    return "\r\n" if "\r\n" in text else "\n"


def should_skip_path(path: Path) -> bool:
    # Skip any file under a skipped directory name
    parts = {p.lower() for p in path.parts}
    return any(skip.lower() in parts for skip in SKIP_DIR_NAMES)


def has_pragma_once(text: str) -> bool:
    return PRAGMA_ONCE_RE.search(text) is not None


def has_existing_guard(text: str) -> bool:
    # Conservative check: if it already uses ifndef/define at top, don't try to re-guard.
    # (You can relax this if needed.)
    top = "\n".join(text.splitlines()[:15])
    return bool(re.search(r"(?m)^\s*#\s*ifndef\b", top) and re.search(r"(?m)^\s*#\s*define\b", top))

In [5]:
def scan_candidates() -> list[Candidate]:
    candidates: list[Candidate] = []

    for path in ROOT.rglob("*"):  # includes files/dirs
        if path.is_dir():
            # quick pruning: don't descend into skipped directories
            if path.name.lower() in {s.lower() for s in SKIP_DIR_NAMES}:
                # rglob can't be pruned directly; we handle skipping at file-level.
                continue
            continue

        if path.suffix.lower() not in HEADER_EXTS:
            continue
        if should_skip_path(path):
            continue

        text = path.read_text(encoding="utf-8", errors="replace")
        if not has_pragma_once(text):
            continue

        rel = path.relative_to(ROOT).as_posix()
        guard = compute_guard(rel)
        candidates.append(Candidate(path=path, rel=rel, guard=guard))

    return candidates


candidates = scan_candidates()
print(f"Found {len(candidates)} headers containing #pragma once")
for c in candidates[:20]:
    print("-", c.rel)
# if len(candidates) > 20:
#     print(f"... and {len(candidates) - 20} more")

Found 77 headers containing #pragma once
- include/Engine/Core/ansi_colors.hpp
- include/Engine/Core/Exceptions.hpp
- include/Engine/Core/Keyboard.hpp
- include/Engine/Core/Mouse.hpp
- include/Engine/Core/utils.hpp
- include/Engine/Core/Window.hpp
- include/Engine/Graphics/Buffer.hpp
- include/Engine/Graphics/CubeShadowMap.hpp
- include/Engine/Graphics/Descriptors.hpp
- include/Engine/Graphics/Device.hpp
- include/Engine/Graphics/DeviceMemory.hpp
- include/Engine/Graphics/FrameBuffer.hpp
- include/Engine/Graphics/FrameInfo.hpp
- include/Engine/Graphics/HZBGenerator.hpp
- include/Engine/Graphics/ImGuiManager.hpp
- include/Engine/Graphics/MorphTargetCompute.hpp
- include/Engine/Graphics/Pipeline.hpp
- include/Engine/Graphics/Renderer.hpp
- include/Engine/Graphics/RenderGraph.hpp
- include/Engine/Graphics/ShadowMap.hpp


In [6]:
def resolve_guard_collisions(cands: list[Candidate]) -> list[Candidate]:
    # If multiple files sanitize to the same macro, add a suffix based on the rel path hash.
    by_guard: dict[str, list[Candidate]] = {}
    for c in cands:
        by_guard.setdefault(c.guard, []).append(c)

    resolved: list[Candidate] = []
    collisions = {g: xs for g, xs in by_guard.items() if len(xs) > 1}

    if collisions:
        print(f"Guard collisions detected: {len(collisions)}")
        for guard, xs in list(collisions.items())[:10]:
            print(guard)
            for x in xs:
                print("  " + x.rel)
        if len(collisions) > 10:
            print("... (truncated)")

    for c in cands:
        if c.guard in collisions:
            suffix = short_hash(c.rel)
            new_guard = f"{c.guard}_{suffix}"
            resolved.append(Candidate(path=c.path, rel=c.rel, guard=new_guard))
        else:
            resolved.append(c)

    return resolved


resolved = resolve_guard_collisions(candidates)
print(f"Resolved set size: {len(resolved)}")

Resolved set size: 77


In [7]:
def rewrite_with_guard(path: Path, guard: str) -> tuple[bool, str]:
    text = path.read_text(encoding="utf-8", errors="replace")
    if not has_pragma_once(text):
        return False, "no pragma once"
    if has_existing_guard(text):
        # Safer: don't try to combine multiple guard styles automatically.
        return False, "already has ifndef/define guard near top"

    nl = detect_newline_style(text)

    # Replace the pragma line (and any immediately following blank lines) with the guard header + one blank line.
    guard_header = f"#ifndef {guard}{nl}#define {guard}{nl}{nl}"
    updated = re.sub(
        r"(?m)^\s*#\s*pragma\s+once\b.*(?:\r?\n)+",
        guard_header,
        text,
        count=1,
    )

    # Ensure file ends with newline
    if not updated.endswith(nl):
        updated += nl

    # Append endif
    endif_line = f"#endif // {guard}{nl}"
    updated += nl + endif_line

    return True, updated


def preview_changes(cands: list[Candidate], limit: int = 15) -> None:
    shown = 0
    for c in cands:
        ok, result = rewrite_with_guard(c.path, c.guard)
        if not ok:
            continue
        shown += 1
        print("\n---")
        print(c.rel)
        print(c.guard)
        # Show first ~12 lines of the updated file
        lines = result.splitlines()
        for line in lines[:12]:
            print(line)
        if shown >= limit:
            break

    if shown == 0:
        print("No eligible files to rewrite (all skipped or already guarded).")


preview_changes(resolved, limit=10)


---
include/Engine/Core/ansi_colors.hpp
VULKANENGINE_INCLUDE_ENGINE_CORE_ANSI_COLORS_HPP
#ifndef VULKANENGINE_INCLUDE_ENGINE_CORE_ANSI_COLORS_HPP
#define VULKANENGINE_INCLUDE_ENGINE_CORE_ANSI_COLORS_HPP

#define RED           "\033"
#define GREEN         "\033"
#define YELLOW        "\033"
#define BLUE          "\033"
#define MAGENTA       "\033"
#define CYAN          "\033"
#define WHITE         "\033"
#define BLACK         "\033"
#define GRAY          "\033"

---
include/Engine/Core/Exceptions.hpp
VULKANENGINE_INCLUDE_ENGINE_CORE_EXCEPTIONS_HPP
#ifndef VULKANENGINE_INCLUDE_ENGINE_CORE_EXCEPTIONS_HPP
#define VULKANENGINE_INCLUDE_ENGINE_CORE_EXCEPTIONS_HPP

#include <exception>
#include <stdexcept>
#include <string>
namespace engine {

  /**
   * @class RuntimeException
   * @brief Generic runtime error used across the engine instead of
   * std::runtime_error

---
include/Engine/Core/Keyboard.hpp
VULKANENGINE_INCLUDE_ENGINE_CORE_KEYBOARD_HPP
#ifndef VULKANENGINE_INCLUDE_ENGINE_CORE_K

In [10]:
# ---- Apply (disabled by default) ----
APPLY = True

# Optional: set to True to create a .bak copy next to every modified file
MAKE_BACKUP = True


def apply_changes(cands: list[Candidate]) -> None:
    modified = 0
    skipped = 0

    for c in cands:
        ok, result = rewrite_with_guard(c.path, c.guard)
        if not ok:
            print(f"[SKIP] {c.rel}: {result}")
            skipped += 1
            continue

        if APPLY:
            if MAKE_BACKUP:
                backup_path = c.path.with_suffix(c.path.suffix + ".bak")
                if not backup_path.exists():
                    backup_path.write_text(c.path.read_text(encoding="utf-8", errors="replace"), encoding="utf-8")
            c.path.write_text(result, encoding="utf-8")

        modified += 1

    mode = "APPLIED" if APPLY else "DRY-RUN"
    print(f"[{mode}] Would modify {modified} files; skipped {skipped} files")


apply_changes(resolved)

[SKIP] include/Engine/Resources/Model.hpp: already has ifndef/define guard near top
[APPLIED] Would modify 76 files; skipped 1 files


In [12]:
# ---- Cleanup backups ----
# Deletes "*.bak" files that may have been created by MAKE_BACKUP.
# Default is preview-only.

DELETE_BAK_FILES = True

def list_bak_files() -> list[Path]:
    return sorted([p for p in ROOT.rglob("*.bak") if not should_skip_path(p)])

bak_files = list_bak_files()
print(f"Found {len(bak_files)} .bak files")
for p in bak_files[:50]:
    print("-", p.relative_to(ROOT).as_posix())
if len(bak_files) > 50:
    print(f"... and {len(bak_files) - 50} more")

if DELETE_BAK_FILES:
    deleted = 0
    for p in bak_files:
        try:
            p.unlink()
            deleted += 1
        except OSError as e:
            print(f"[FAILED] {p}: {e}")
    print(f"Deleted {deleted} .bak files")
else:
    print("Preview mode: set DELETE_BAK_FILES = True to delete.")

Found 76 .bak files
- include/Engine/Core/ansi_colors.hpp.bak
- include/Engine/Core/Exceptions.hpp.bak
- include/Engine/Core/Keyboard.hpp.bak
- include/Engine/Core/Mouse.hpp.bak
- include/Engine/Core/utils.hpp.bak
- include/Engine/Core/Window.hpp.bak
- include/Engine/Graphics/Buffer.hpp.bak
- include/Engine/Graphics/CubeShadowMap.hpp.bak
- include/Engine/Graphics/Descriptors.hpp.bak
- include/Engine/Graphics/Device.hpp.bak
- include/Engine/Graphics/DeviceMemory.hpp.bak
- include/Engine/Graphics/FrameBuffer.hpp.bak
- include/Engine/Graphics/FrameInfo.hpp.bak
- include/Engine/Graphics/HZBGenerator.hpp.bak
- include/Engine/Graphics/ImGuiManager.hpp.bak
- include/Engine/Graphics/MorphTargetCompute.hpp.bak
- include/Engine/Graphics/Pipeline.hpp.bak
- include/Engine/Graphics/Renderer.hpp.bak
- include/Engine/Graphics/RenderGraph.hpp.bak
- include/Engine/Graphics/ShadowMap.hpp.bak
- include/Engine/Graphics/SwapChain.hpp.bak
- include/Engine/Graphics/VulkanConfig.hpp.bak
- include/Engine/Resou